## Setup & Imports

In [ ]:
%pip install langchain-openai pandas faiss-cpu google-api-python-client langchain-community

## Data Collection : Load books from Google Books API


In [ ]:
import os
import time
from datetime import datetime
from typing import List, Dict
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.document_loaders.base import BaseLoader
import requests
from tqdm import tqdm
import pandas as pd


load_dotenv()

In [70]:
class SystematicGoogleBooksLoader(BaseLoader):
    """Loader that fetches books per search value systematically"""
    
    def __init__(self):
        self.BASE_URL = "https://www.googleapis.com/books/v1/volumes"
        self.RESULTS_PER_PAGE = 10
        self.MIN_BOOKS_PER_VALUE = 5
        self.MIN_DESCRIPTION_LENGTH = 2048
        self.API_KEY = os.getenv("GOOGLE_API_KEY")
        
        self.SEARCH_STRATEGIES = [
            {
                "param": "subject", 
                "values": [ "Fiction", "Science", "History", "Biography", "Technology", "Art", "Philosophy", "Psychology", "Business", "Travel"]
            },
            {
                "param": "inauthor", 
                "values": ["Stephen King", "J.K. Rowling", "Yuval Noah Harari", "Michelle Obama", "Malcolm Gladwell", "J.R.R. Tolkien", "Agatha Christie", "Toni Morrison"]
            },
            {
                "param": "intitle", 
                "values": [ "Python", "Machine Learning", "World War", "Artificial Intelligence", "Space Exploration", "Climate Change", "Ancient History", "Mental Health"]
            },
            {
                "param": "intitle", 
                "values": [ "Python", "Machine Learning", "World War", "Artificial Intelligence", "Space Exploration", "Climate Change", "Ancient History", "Mental Health"]
            }
        ]
        
        self.books_data = {}  # book_id -> book_data
        self.value_counts = {}  # (param, value) -> count

    def load(self) -> List[Document]:
        """Fetch books systematically for all strategy values"""
        print(f"Fetching at least {self.MIN_BOOKS_PER_VALUE} books per search value...")
        
        # Initialize counters
        for strategy in self.SEARCH_STRATEGIES:
            for value in strategy["values"]:
                self.value_counts[(strategy["param"], value)] = 0
        
        # Process each search value systematically
        for strategy in tqdm(self.SEARCH_STRATEGIES, desc="Processing strategies"):
            param = strategy["param"]
            for value in tqdm(strategy["values"], desc=f"Processing {param} values"):
                self._fetch_for_value(param, value)
                time.sleep(1)
        
        print(f"\nTotal books collected: {len(self.books_data)}")
        return pd.DataFrame(self.books_data.values())

    def _fetch_for_value(self, param: str, value: str) -> None:
        """Fetch books for a specific parameter value until we have enough"""
        query = f"{param}:{value}"
        start_index = 0
        collected = 0
        
        while collected < self.MIN_BOOKS_PER_VALUE:
            try:
                response = requests.get(
                    self.BASE_URL,
                    params={
                        "q": query,
                        "maxResults": self.RESULTS_PER_PAGE,
                        "startIndex": start_index,
                        "key": self.API_KEY,
                        "printType": "books",
                        "langRestrict": "en"
                    },
                    timeout=15
                )
                response.raise_for_status()
                data = response.json()
                
                if "items" not in data:
                    print(f"\nNo more results for '{query}'")
                    break
                
                new_books = self._process_items(data["items"], param, value)
                collected += new_books
                start_index += self.RESULTS_PER_PAGE
                
                # Break if API returns fewer items than requested
                if len(data.get("items", [])) < self.RESULTS_PER_PAGE:
                    break
                    
                time.sleep(1)  # Rate limiting between API calls
                
            except requests.exceptions.RequestException as e:
                print(f"\nAPI Error for '{query}': {str(e)}")
                time.sleep(5)
                continue
        
        print(f"\nCollected {collected} books for {param}:{value}")

    def _process_items(self, items: List[Dict], param: str, value: str) -> int:
        """Process API items and return count of new books added"""
        new_books = 0
        for item in items:
            volume_info = item.get("volumeInfo", {})
            book_id = item.get("id")
            
            # Skip if doesn't meet criteria
            if (not volume_info.get("description") or 
                len(volume_info["description"]) < self.MIN_DESCRIPTION_LENGTH or
                book_id in self.books_data):
                continue
            
            # Store book data
            self.books_data[book_id] = {
                "id": book_id,
                "title": volume_info.get("title", "Untitled"),
                "authors": ", ".join(volume_info.get("authors", ["Unknown"])),
                "description": volume_info.get("description", ""),
                "categories": ", ".join(volume_info.get("categories", [])),
                "primary_category": volume_info.get("categories", ["General"])[0],
               
                "search_strategy": f"{param}:{value}",
                "retrieval_timestamp": datetime.now().isoformat()
            }
            self.value_counts[(param, value)] += 1
            new_books += 1
        
        return new_books

    def get_coverage_report(self) -> Dict:
        """Generate report showing books collected per search value"""
        report = {}
        for strategy in self.SEARCH_STRATEGIES:
            param = strategy["param"]
            report[param] = {}
            for value in strategy["values"]:
                report[param][value] = self.value_counts.get((param, value), 0)
        return report
    
    def save_to_csv(self, filename: str) -> None:
        """Save books data to CSV file"""
        df = pd.DataFrame(self.books_data.values())
        df.to_csv(filename, index=False)

In [ ]:
loader = SystematicGoogleBooksLoader()
df_books = loader.load()
report = loader.get_coverage_report()

In [ ]:
df_books

In [ ]:
print("\nSearch Strategy Coverage:")
for param, values in report.items():
    print(f"\n{param.upper()}:")
    for value, count in values.items():
        print(f"  {value}: {count} books")

In [ ]:
print("\nSample document:")
sample = df_books[0]
print(f"  Title: {sample.metadata['title']}")
print(f"  Author: {sample.metadata['authors']}")
print(f"  Categories: {sample.metadata['categories']}")
print(f"  Description: {sample.page_content.split('Description: ')[1][:100]}...")

## Data Processing using Pandas

In [32]:
class DataProccessor:

    def __init__(self, df: pd.DataFrame):
        self.df = df

    def describe_data(self):
        print(f"Dataset shape: {self.df.shape}")
        print(f"Columns: {list(self.df.columns)}")
        print("First few rows:")
        print(self.df.head())

    def clean_data(self):
        """ Clean the data description by removing duplicates and missing values and whitespace"""

        self.df = self.df.dropna(subset=["description"])
        self.df = self.df.drop_duplicates(subset=["description"])
        self.df = self.df.dropna(subset=["title"])
        self.df = self.df.drop_duplicates(subset=["title"])
        self.df["description"] = self.df["description"].str.replace(r'\s+', ' ', regex=True)
        self.df["description"] = self.df["description"].str.strip()
        self.df = self.df[self.df["description"] != ""]
        return self.df



In [35]:
Processor = DataProccessor(pd.DataFrame(documents.page_content))
documents.page_content = Processor.clean_data()

In [ ]:
documents.describe_data()

## Embedding Book Descriptions using OpenAIEmbeddings Model

In [57]:
df = pd.read_csv("data/books_20250703_225726.csv")

In [ ]:

from langchain_openai import OpenAIEmbeddings
import ast


embeddings_model = OpenAIEmbeddings( model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))

# Batch processing for efficiency
batch_size = 50
embeddings = []
for i in range(0, len(df), batch_size):
    batch = df["description"].iloc[i:i+batch_size].tolist()
    embeddings.extend(embeddings_model.embed_documents(batch))
    
df["embedding"] = embeddings
df["embedding"] = df["embedding"].apply(lambda x: [float(i) for i in x])

df.to_csv("data/books_embeddings.csv")
print(len(df))

## Storing Data using FAISS Vector Store

In [59]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_embeddings(
    text_embeddings=zip(df["description"], df["embedding"]),
    embedding=embeddings_model,
    metadatas=df.drop(columns=["description", "embedding"]).to_dict("records")
)
vector_store.save_local("book_index")

## Similarity Search and Recommendation

Cosine Similarity = (A · B) / (||A|| * ||B||)

In [64]:
def recommend_books(query, k=10):
    # Get similar documents
    docs = vector_store.similarity_search_with_relevance_scores(query, k=k)
    
    # Convert to Pandas DataFrame
    results = []
    for doc, score in docs:
        meta = doc.metadata
        results.append({
            "id": meta["id"],
            "title": meta["title"],
            "authors": meta["authors"],
            "description": doc.page_content,
            "similarity_score": score
        })
    return pd.DataFrame(results)

In [ ]:
recommend_books("europe history", k=10)

## Performance Assessment: Relevance Metrics

In [ ]:
df_embeddings.iloc[1]

In [ ]:
df_embeddings["categories"].unique()

In [ ]:
history_df = df_embeddings[df_embeddings["categories"] == "History"]
history_df["id"]

In [ ]:
len(df_embeddings[df_embeddings["categories"] == "History"])

In [73]:
def calculate_metrics(results, ground_truth):
    relevant_count = sum(1 for doc in results if doc.metadata['id'] in ground_truth)
    precision = relevant_count / len(results)
    recall = relevant_count / len(ground_truth)
    return precision, recall


In [ ]:
df_embeddings[df_embeddings["id"] == "qKQd3S1-vEsC"]["id"]

In [81]:
ground_truth_ids = list(history_df["id"])

results = vector_store.similarity_search(str(df_embeddings[df_embeddings["id"] == "qKQd3S1-vEsC"]["id"]), k=20)
precision, recall = calculate_metrics(results, ground_truth_ids)

In [ ]:
precision, recall